# Gemini-3-Flash-Preview CC

In [3]:
# Cell 1: Setup (Gemini planner pipeline for coin-collector data, independent)
import os
import re
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/planner_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Data / prompt / output paths (coin-collector data)
CC_DATA_DIR = PROJECT_ROOT / "cc_data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "llm_planner_instruction_cc.txt"
DOMAIN_PDDL_PATH = CC_DATA_DIR / "pddl" / "domain.pddl"

MODEL_DIRNAME = "gemini_3_flash"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner_cc" / MODEL_DIRNAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# VAL executable
VAL_PATH = Path.home() / "tools" / "VAL" / "build" / "linux64" / "Release" / "bin" / "Validate"

# Gemini model + API key
MODEL_NAME = "gemini-3-flash-preview"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError(f"GOOGLE_API_KEY not found in {ENV_PATH}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CC_DATA_DIR:", CC_DATA_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
CC_DATA_DIR: /home/sagemaker-user/formalizer-scaling/cc_data
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/llm_planner_cc/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [4]:
# Cell 2: Load problem IDs, planner instruction, and ground-truth domain PDDL
import pandas as pd

# Use all available problems by default; edit this list later for targeted runs
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 101)]

PLANNER_INSTRUCTION = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Planner prompt chars:", len(PLANNER_INSTRUCTION))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 100
First/last: p01 p100
Planner prompt chars: 1281
Domain PDDL chars: 679


In [5]:
# Cell 3: Helper - load NL domain/problem descriptions for one problem ID
def load_descriptions(problem_id: str):
  domain_desc_path = CC_DATA_DIR / "descriptions" / f"{problem_id}_domain.txt"
  problem_desc_path = CC_DATA_DIR / "descriptions" / f"{problem_id}_problem.txt"

  if not domain_desc_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_desc_path}")
  if not problem_desc_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_desc_path}")

  return domain_desc_path.read_text(), problem_desc_path.read_text()

In [6]:
# Cell 4: Helper - build planner prompt (NL domain + NL problem + GT domain PDDL)
def build_prompt(problem_id: str):
  domain_desc, problem_desc = load_descriptions(problem_id)

  return (
      PLANNER_INSTRUCTION
      + "\n\nYou are an expert automated planner.\n"
      "Create an executable plan that is consistent with all provided information.\n\n"
      "### Natural-language domain description\n"
      f"{domain_desc}\n\n"
      "### Natural-language problem description\n"
      f"{problem_desc}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output only the final plan wrapped in <plan>...</plan>."
  )

In [7]:
# Cell 5: Helper - call Gemini for direct plan generation
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

def call_gemini(prompt: str):
  resp = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0.4,
          max_output_tokens=200000,
      ),
  )
  return resp.text or ""

In [8]:
# Cell 6: Helper - extract <plan>...</plan> block
def extract_plan(raw_text: str):
  if not raw_text:
      return ""

  m = re.search(r"<plan>(.*?)</plan>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if not m:
      return ""

  plan_text = m.group(1).strip()

  # Keep one action per non-empty line
  lines = [ln.strip() for ln in plan_text.splitlines() if ln.strip()]
  return "\n".join(lines)

In [9]:
# Cell 7: Helper - validate a plan with VAL (GT domain + GT problem)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [10]:
# Cell 8: Helper - run one problem end-to-end (generate plan + VAL), persist outputs
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)
  plan_text = extract_plan(raw_text)

  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Always persist raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Persist GT domain file for reproducibility of validation context
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  if not plan_text.strip():
      # No plan extracted
      (cand_dir / "planner_error.txt").write_text("No <plan>...</plan> found in model output.")
      return {
          "problem": problem_id,
          "plan_extracted": False,
          "val_ok": False,
          "planner_error": "No <plan>...</plan> found in model output.",
      }

  # Save plan and run VAL against GT problem
  plan_path = cand_dir / "plan.txt"
  plan_path.write_text(plan_text)

  gt_problem_path = CC_DATA_DIR / "pddl" / f"{problem_id}.pddl"
  if not gt_problem_path.exists():
      msg = f"Missing ground-truth problem file: {gt_problem_path}"
      (cand_dir / "planner_error.txt").write_text(msg)
      return {
          "problem": problem_id,
          "plan_extracted": True,
          "val_ok": False,
          "planner_error": msg,
      }

  val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)
  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return {
      "problem": problem_id,
      "plan_extracted": True,
      "val_ok": bool(val_result.get("val_ok", False)),
      "planner_error": "",
  }

In [14]:
# Cell 9: Generate only (no VAL)
smoke_problem = "p145"

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
plan_text = extract_plan(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "plan.txt").write_text(plan_text or "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_plan_chars:", len(plan_text or ""))
print("saved:", cand_dir)

Generation-only done.
problem: p145
extracted_plan_chars: 65
saved: /home/sagemaker-user/formalizer-scaling/output/llm_planner_cc/gemini_3_flash/p145/cand_00


In [25]:
# Cell 10: VAL only (no model call), concise output
import contextlib
import io

smoke_problem = "p150"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

plan_path = cand_dir / "plan.txt"
if not plan_path.exists() or not plan_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty plan file: {plan_path}")

gt_problem_path = CC_DATA_DIR / "pddl" / f"{smoke_problem}.pddl"
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

with contextlib.redirect_stdout(io.StringIO()):
  val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)

(cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

summary = {
  "problem": smoke_problem,
  "plan_extracted": True,
  "val_ok": bool(val_result.get("val_ok", False)),
}

print("Produced files:")
print("-", "plan.txt")
print("-", "validator.log")
print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
print("Summary:")
print(json.dumps(summary, indent=2))

Produced files:
- plan.txt
- validator.log
VAL result: valid
Summary:
{
  "problem": "p150",
  "plan_extracted": true,
  "val_ok": true
}


In [15]:
# Cell 11: Batch generate plans only (no VAL)
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(146, 151)]  # change as needed

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
  try:
      prompt = build_prompt(pid)
      raw_text = call_gemini(prompt)
      plan_text = extract_plan(raw_text)

      cand_dir = OUTPUT_ROOT / pid / "cand_00"
      cand_dir.mkdir(parents=True, exist_ok=True)

      (cand_dir / "raw.txt").write_text(raw_text or "")
      (cand_dir / "plan.txt").write_text(plan_text or "")
      (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

      ok = bool(plan_text.strip())
      gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})
      print(f"{pid}: generation {'ok' if ok else 'failed (no <plan>)'}")
  except Exception as e:
      gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
      print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r["generation_ok"]))
print("failed:", sum(1 for r in gen_results if not r["generation_ok"]))

LLM generation:   0%|          | 0/5 [00:00<?, ?problem/s]

p146: generation ok
p147: generation ok
p148: generation ok
p149: generation ok
p150: generation ok

Generation batch done.
total: 5
ok: 5
failed: 0


In [ ]:
# Cell 12: Batch VAL only (no model call)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_VAL = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(91, 101)]  # change as needed

val_results = []
for pid in tqdm(BATCH_PROBLEMS_VAL, desc="VAL", unit="problem"):
  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  plan_path = cand_dir / "plan.txt"
  gt_problem_path = CC_DATA_DIR / "pddl" / f"{pid}.pddl"

  if not plan_path.exists() or not plan_path.read_text().strip():
      msg = f"missing/empty plan file: {plan_path}"
      val_results.append({"problem": pid, "val_ok": False, "error": msg})
      print(f"{pid}: skipped ({msg})")
      continue

  if not gt_problem_path.exists():
      msg = f"missing GT problem file: {gt_problem_path}"
      val_results.append({"problem": pid, "val_ok": False, "error": msg})
      print(f"{pid}: skipped ({msg})")
      continue

  with contextlib.redirect_stdout(io.StringIO()):
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  ok = bool(val_result.get("val_ok", False))
  val_results.append({"problem": pid, "val_ok": ok, "error": ""})

  print(f"{pid}: {'valid' if ok else 'invalid'}")

print("\nVAL batch done.")
print("total:", len(val_results))
print("valid:", sum(1 for r in val_results if r["val_ok"]))
print("invalid_or_failed:", sum(1 for r in val_results if not r["val_ok"]))

In [ ]:
# Cell 13: In-notebook metrics summary (no persistence)
import pandas as pd

# Define any batch you want here
SUMMARY_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(91, 101)]  # example batch

rows = []
for pid in SUMMARY_PROBLEMS:
  cand_dir = OUTPUT_ROOT / pid / "cand_00"

  raw_path = cand_dir / "raw.txt"
  plan_path = cand_dir / "plan.txt"
  val_log_path = cand_dir / "validator.log"
  planner_error_path = cand_dir / "planner_error.txt"

  raw_text = raw_path.read_text() if raw_path.exists() else ""
  plan_text = plan_path.read_text() if plan_path.exists() else ""
  val_log = val_log_path.read_text() if val_log_path.exists() else ""
  planner_error = planner_error_path.read_text().strip() if planner_error_path.exists() else ""

  plan_extracted = bool(plan_text.strip())
  val_ran = bool(val_log.strip())
  val_ok = "Plan valid" in val_log

  if val_ok:
      status = "plan_valid"
  elif plan_extracted and val_ran:
      status = "plan_invalid"
  elif planner_error:
      status = "planner_failed"
  else:
      status = "unknown_or_not_run"

  rows.append(
      {
          "problem_id": pid,
          "status": status,
          "plan_extracted": plan_extracted,
          "val_ran": val_ran,
          "val_ok": val_ok,
          "planner_error": planner_error,
          "raw_chars": len(raw_text),
          "plan_chars": len(plan_text),
          "cand_dir": str(cand_dir),
      }
  )

details_df = pd.DataFrame(rows).sort_values("problem_id").reset_index(drop=True)

total = len(details_df)
plan_extracted_n = int(details_df["plan_extracted"].sum())
val_ok_n = int(details_df["val_ok"].sum())
planner_failed_n = int((details_df["status"] == "planner_failed").sum())
plan_invalid_n = int((details_df["status"] == "plan_invalid").sum())

summary_df = pd.DataFrame(
  [
      {
          "num_problems": total,
          "plan_extracted_count": plan_extracted_n,
          "plan_extracted_rate": plan_extracted_n / total if total else 0.0,
          "val_ok_count": val_ok_n,
          "val_ok_rate": val_ok_n / total if total else 0.0,
          "planner_failed_count": planner_failed_n,
          "plan_invalid_count": plan_invalid_n,
      }
  ]
)

print("Summary:")
display(summary_df)

print("Per-problem details:")
display(details_df)

# Keep these for optional persistence in Cell 14
SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS

In [ ]:
# Cell 14: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

# Optional label to distinguish runs (edit as needed)
SAVE_LABEL = "gemini_3_flash_cc_batch"

# Where to save metric CSVs
METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
  raise RuntimeError("Run Cell 13 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = (
  f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}"
  if len(SUMMARY_PROBLEMS_USED) > 1
  else SUMMARY_PROBLEMS_USED[0]
)

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)